We load dependencies


In [ ]:
! pip install openai chromadb

fowchart how rag will work,
1. we need a embedding function for setting up database with implicit embedding for our questions asked
2. we'll setup the database and ensure we pass our embedding function created
3. we will load our documents and store in a list with ids: as filename(unique), text as text content of the document
4. we will chunk our document and maske sure to overlap the chunks so that consistency is maintained
5. we will make another list and store the ids and the chunks for each document very carefully so that a document id is same for all its chunks
6. we will write an embedding function to embed the chunks and store in new list embeded chunks with ids, text, embeddings
7. we will create a function to extract relevent chunks for the qeustion
8. we will call our assistant with relevent document, pompt for making it understand it's task and add the relevant chunk here, and question to get its answer

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata
import chromadb
from chromadb.utils import embedding_functions

#API key initializing
api_key = userdata.get('raj_api_key')

#openai embedding function setup for chromadb
openai_ef= embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)

#settingup chromadb
chroma_client=chromadb.PersistentClient(path='/content/sample_data/chroma_persistant_storage')
collection_name='doucument_qa_collection'
collection=chroma_client.get_or_create_collection(name=collection_name, embedding_function=openai_ef)


# loading documents from directory and storing it with content and ids as a list of dictionary
def load_documents(directory_path):
  document=[]
  for filename in os.listdir(directory_path):
    filepath = os.path.join(directory_path, filename)
    if os.path.isfile(filepath):
      with open(filepath, 'r', encoding='utf-8') as file:
        document.append({'id':filename, 'content':file.read()})
  return document


directory_path = "/content/sample_data/rag_news_articles"
documents = load_documents(directory_path)

#dividing the text into chunks
def split_text(text, chunk_size=1000, chunk_overlap=20):
    chunk=[]
    start=0
    while start<len(text):
      end=start+chunk_size
      chunk.append(text[start:end])
      start=end-chunk_overlap
    return chunk

#dividing the document into cunks and storing it in new list as chunked_documents
chunked_document=[]
for doc in documents:
  chunks=split_text(doc.get('content'))
  for i, chunk in enumerate(chunks):
    chunked_document.append({'id':f'{doc.get('id')}_chunk_{i+1}', 'text':chunk})


#upload the embedded document into chromadb
for doc in chunked_document:
  collection.upsert(ids=[doc.get('id')], documents=[doc.get('text')])


#fetch relevent chunks for question
def fetch_relevent_chunks(question, n=2):
  relevent_chunks=[]
  response=collection.query(query_texts=[question], n_results=n)
  relevent_chunks=[doc for sublist in response.get('documents') for doc in sublist]
  return relevent_chunks

#assistant calling
def assistant(question, relevent_chunks):
  client = OpenAI(api_key=api_key)
  context_string = "\n".join(relevent_chunks)

  system_prompt = (
      "You are an assistant for question-answering tasks. Use the following pieces of "
      "retrieved context to answer the question. If you don't know the answer, say that you "
      "don't know. Use three sentences maximum and keep the answer concise."
  )

  user_message = f"\n\nContext:\n{context_string}\n\nQuestion:\n{question}"

  response = client.chat.completions.create(
      model="gpt-3.5-turbo",
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_message}
      ]
  )
  return response.choices[0].message.content

def chatbot():
  print("Welcome to the chatbot! Type 'quit' to exit.")
  while True:
    user_input=input("User: ")
    print()
    if user_input.lower()=='quit' or 'stop':
      break
    relevant_chunks=fetch_relevent_chunks(user_input)
    answer=assistant(user_input, relevant_chunks)
    print("Chatbot:", answer.replace('.', '.\n'))
    print()

chatbot()